In [56]:
# 1. Imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    confusion_matrix
)

from xgboost import XGBClassifier


In [57]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")

print("Path to dataset files:", path)

df = pd.read_csv(os.path.join(path, "AB_NYC_2019.csv"))


Path to dataset files: C:\Users\HP\.cache\kagglehub\datasets\dgomonov\new-york-city-airbnb-open-data\versions\3


In [58]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [59]:
df.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


In [60]:
df.isnull().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     38843 non-n

In [62]:
df.columns.unique()

Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
       'neighbourhood', 'latitude', 'longitude', 'room_type', 'price',
       'minimum_nights', 'number_of_reviews', 'last_review',
       'reviews_per_month', 'calculated_host_listings_count',
       'availability_365'],
      dtype='str')

In [63]:
df.value_counts()

id        name                                              host_id    host_name    neighbourhood_group  neighbourhood    latitude  longitude  room_type        price  minimum_nights  number_of_reviews  last_review  reviews_per_month  calculated_host_listings_count  availability_365
2539      Clean & quiet apt home by the park                2787       John         Brooklyn             Kensington       40.64749  -73.97237  Private room     149    1               9                  2018-10-19   0.21               6                               365                 1
2595      Skylit Midtown Castle                             2845       Jennifer     Manhattan            Midtown          40.75362  -73.98377  Entire home/apt  225    1               45                 2019-05-21   0.38               2                               355                 1
3831      Cozy Entire Floor of Brownstone                   4869       LisaRoxanne  Brooklyn             Clinton Hill     40.68514  -73.95976

DATA CLEANING


In [64]:
df_clean = df.drop(columns=['id', 'name', 'host_id', 'host_name', 'last_review']).copy()
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)

required_columns = ['room_type', 'price', 'minimum_nights']
missing_columns = [c for c in required_columns if c not in df_clean.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Normalise target
for c in ['neighbourhood_group', 'neighbourhood', 'room_type']: df_clean[c]= df_clean[c].astype("string").str.strip()

In [65]:
##OUTLIER
price_cap = df_clean['price'].quantile(0.99)
nights_cap = df_clean['minimum_nights'].quantile(0.99)

df_clean['price'] = df_clean['price'].clip(upper=price_cap)
df_clean['minimum_nights'] = df_clean['minimum_nights'].clip(upper=nights_cap)


In [ ]:
X = df_clean.drop(columns=['room_type'])
y = df_clean['room_type']
if y.isna().any(): raise ValueError("Unexpected House labels found.")

In [67]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_clean["room_type"])

In [68]:
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.30,stratify=y,random_state=42)
X_valid,X_test,y_valid,y_test=train_test_split(X_temp,y_temp,test_size=.50,stratify=y_temp,random_state=42)

In [69]:
num=X.select_dtypes(include=["number"]).columns.tolist()
cat=X.select_dtypes(exclude=["number"]).columns.tolist()
num, cat

(['latitude',
  'longitude',
  'price',
  'minimum_nights',
  'number_of_reviews',
  'reviews_per_month',
  'calculated_host_listings_count',
  'availability_365'],
 ['neighbourhood_group', 'neighbourhood'])

In [70]:
numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="median")),
    ('scale', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))

])
preprocessor = ColumnTransformer(transformers=[
    ("numerical", numeric_pipeline, num),
    ('categorical', categorical_pipeline, cat)
])
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [71]:
df_clean[num].skew()

latitude                          0.237167
longitude                         1.284210
price                             2.776321
minimum_nights                    2.360347
number_of_reviews                 3.690635
reviews_per_month                 3.300723
calculated_host_listings_count    7.933174
availability_365                  0.763408
dtype: float64

In [72]:
models={
"LogisticRegression":LogisticRegression(max_iter=3000,class_weight="balanced",random_state=42),
"Randcl":RandomForestClassifier(random_state=41, class_weight='balanced'),
"XGBoost":XGBClassifier(learning_rate=0.2,objective='multi:softprob',random_state=42,n_jobs=-1, num_class='3'),
'Hist':HistGradientBoostingClassifier()
}


In [73]:
rows=[]; fitted={}

for name,m in models.items():
 
 p=Pipeline([("preprocess",preprocessor),("model",m)]);
 accuracy= cross_val_score(p, X_train, y_train, cv=4, scoring='accuracy')
 macrof1 = cross_val_score(p, X_train, y_train, cv=4, scoring='f1_macro')
 
 print(f'{name} => Accuracy {accuracy.mean():.3f}, F1: {macrof1.mean():.3f}')

LogisticRegression => Accuracy 0.658, F1: 0.521
Randcl => Accuracy 0.854, F1: 0.736
XGBoost => Accuracy 0.859, F1: 0.734
Hist => Accuracy 0.859, F1: 0.737


In [74]:
rows=[]; fitted={}

for name,m in models.items():
 
 p=Pipeline([("preprocess",preprocessor),("model",m)]); 
 p.fit(X_train,y_train); 
 prob=p.predict_proba(X_valid); 
 pred = prob.argmax(axis=1)
 fitted[name] = p
 rows.append([
    name,
    accuracy_score(y_valid, pred),
    balanced_accuracy_score(y_valid, pred),
    roc_auc_score(y_valid, prob, multi_class="ovr", average="macro"),
    f1_score(y_valid, pred, average="macro", zero_division=0)
 ])
results=pd.DataFrame(rows,columns=["model","accuracy","balanced_accuracy","roc_auc","f1"]).sort_values(["balanced_accuracy","roc_auc","f1"],ascending=False)

In [75]:
best_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', XGBClassifier(
        learning_rate=0.2,
        objective='multi:softprob',
        random_state=42,
        n_jobs=-1,
        num_class=3,
        eval_metric='mlogloss'
    ))
])

param_dist = {
    'classifier__n_estimators': [100, 200, 300, 400, 500, 600, 800, 1000],
    'classifier__max_depth': [3, 4, 5, 6, 7, 8, 10],
    'classifier__subsample': np.linspace(0.6, 1.0, 5),
    'classifier__colsample_bytree': np.linspace(0.5, 1.0, 6),
    'classifier__gamma': np.linspace(0, 5, 6),
    'classifier__min_child_weight': [1, 3, 5, 7],
    'classifier__reg_alpha': np.linspace(0, 1, 5),
    'classifier__reg_lambda': np.linspace(0.5, 2, 4),
    'classifier__scale_pos_weight': [1, 5, 10, 20]
}

search = RandomizedSearchCV(
    estimator=best_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1

)
search.fit(X_train,y_train)

print(search.best_score_)
print(search.best_params_)




0.7448206928542601
{'classifier__subsample': np.float64(0.8), 'classifier__scale_pos_weight': 5, 'classifier__reg_lambda': np.float64(2.0), 'classifier__reg_alpha': np.float64(0.75), 'classifier__n_estimators': 100, 'classifier__min_child_weight': 3, 'classifier__max_depth': 10, 'classifier__gamma': np.float64(0.0), 'classifier__colsample_bytree': np.float64(1.0)}


In [ ]:
def metrics(model, Xp, yp):
    prob = model.predict_proba(Xp)
    pred = prob.argmax(axis=1)
    return {
        "accuracy": accuracy_score(yp, pred),
        "balanced_accuracy": balanced_accuracy_score(yp, pred),
        "roc_auc": roc_auc_score(yp, prob, multi_class="ovr", average="macro"),
        "precision": precision_score(yp, pred, average="macro", zero_division=0),
        "recall": recall_score(yp, pred, average="macro", zero_division=0),
        "f1": f1_score(yp, pred, average="macro", zero_division=0),
    }, prob, pred

best_model_name = results.iloc[0].model
base = fitted.get(best_model_name)
if base is None:
    raise KeyError(f"Model '{best_model_name}' not found in 'fitted'. Run the model comparison cell before this one.")

a, _, _ = metrics(base, X_valid, y_valid)
b, _, _ = metrics(search.best_estimator_, X_valid, y_valid)
display(pd.DataFrame([a, b], index=["initial_best", "tuned_model"]))

selected = search.best_estimator_ if (
    b["balanced_accuracy"], b["roc_auc"]
) >= (
    a['balanced_accuracy'], b['roc_auc']
) else base


,accuracy,balanced_accuracy,roc_auc,precision,recall,f1
initial_best,0.865694,0.688544,0.926271,0.834431,0.688544,0.728660
tuned_model,0.868285,0.692372,0.924263,0.852237,0.692372,0.734911


In [77]:
with open("final_airbnb_model.pkl", "wb") as f:
    pickle.dump(selected, f)